# Legacy published pipeline dataset (historical inspection only)

The `steam-insights` CLI has been retired. This notebook reads an already published legacy snapshot for historical inspection only. Do not treat its candidates or silver labels as inputs to the replacement system.


In [ ]:
from pathlib import Path
import os

# Find the repository when Jupyter starts in notebooks/.
repo = Path.cwd() if (Path.cwd() / "configs/pipeline.toml").exists() else Path.cwd().parent
os.chdir(repo)
import duckdb
from steam_insights.pipeline.settings import load_pipeline_config
from steam_insights.pipeline.loader import latest_dataset, iter_model_batches
config = load_pipeline_config()
directory = latest_dataset(config)
connection = duckdb.connect(str(directory / "catalog.duckdb"), read_only=True)
connection.execute("SET TimeZone = 'UTC'")


In [ ]:
connection.sql("SELECT game, split, count(*) AS reviews, count(*) FILTER (WHERE text_eligible) AS eligible FROM reviews GROUP BY game, split ORDER BY game").df()


In [ ]:
connection.sql("SELECT verification_status, count(*) AS patch_posts FROM patches GROUP BY verification_status").df()


In [ ]:
connection.sql("SELECT game, title, pre_eligible_reviews, post_eligible_reviews, window_coverage_evidence, coverage_screen_met FROM patch_windows WHERE verified_patch_post ORDER BY coverage_screen_met DESC, game").df()


## Load model input batches

These are candidate inputs, not adjudicated training labels. The supervised view is empty until valid human annotations are imported. Model-specific tokenization belongs downstream.


In [ ]:
batch = next(iter_model_batches(directory, kind="pairs", split="train", batch_size=32), None)
{"rows": 0 if batch is None else batch.num_rows, "fields": [] if batch is None else batch.schema.names}


In [ ]:
connection.close()
